# V4 Prior Optimization: Training the Dixon-Coles Models

This notebook connects to the `v4_historical_data.sqlite` database you just populated with `soccerdata`.
It splits the massive 7,171-match database by League, and feeds each league's history into our custom `continuous_dixon_coles_nll` Maximum Likelihood Estimator to calculate the optimal Attack ($\alpha$) and Defense ($\beta$) parameters for every team.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import json
from pathlib import Path
import sys

# Ensure we can import from the backend directory regardless of cwd
sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd() / 'v4_backend'))
sys.path.append(str(Path.cwd().parent.parent / 'v4_backend'))

from dixon_coles_xg import train_dixon_coles_prior

possible_db_paths = [
    Path("../../v4_historical_data.sqlite"), # If running from v4_backend/notebooks/
    Path("v4_historical_data.sqlite"),       # If running from root
    Path("../v4_historical_data.sqlite")     # If running from v4_backend/
]

DB_PATH = next((p for p in possible_db_paths if p.exists()), None)
if not DB_PATH:
    print("❌ Could not find v4_historical_data.sqlite in expected locations.")
else:
    print(f"✅ Found database at: {DB_PATH.resolve()}")


✅ Found database at: C:\Users\rhkha\Documents\Documents\Schoolwork\Projects\FIFA-WORLDCUP-PREDICTION\v4_historical_data.sqlite


## 1. Load and Prep the Historical Ledger
We need to calculate `days_ago` to apply the exponential time-decay weighting.

In [2]:
try:
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("SELECT * FROM matches_xg", conn)
    conn.close()
    print(f"✅ Successfully loaded {len(df)} matches from SQLite database.")
except Exception as e:
    print(f"❌ Failed to load database: {e}")
    df = pd.DataFrame()

if not df.empty:
    # Convert date strings to datetime objects
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    
    # Drop any rows where dates failed to parse
    df = df.dropna(subset=['date'])
    
    # Calculate days_ago for the time decay function
    # We use the most recent match in the dataset as "today"
    max_date = df['date'].max()
    df['days_ago'] = (max_date - df['date']).dt.days
    
    print(f"Data spans from {df['date'].min().date()} to {max_date.date()}.")


✅ Successfully loaded 7171 matches from SQLite database.
Data spans from 2021-08-06 to 2025-05-29.


## 2. Optimize the Parameters per League
We loop through the 5 Big Leagues, dynamically mapping the team names to numeric indices so the Scipy SLSQP optimizer can vector-map them efficiently.

In [3]:
v4_priors = {}

if not df.empty:
    leagues = df['league'].unique()
    
    for lg in leagues:
        print(f"\n{'='*50}")
        print(f"📈 Optimizing Parameters for: {lg}")
        print(f"{'='*50}")
        
        # Filter matches for this specific league
        df_lg = df[df['league'] == lg].copy()
        
        # Get unique teams in this league
        teams_list = sorted(list(set(df_lg['home_team'].unique()) | set(df_lg['away_team'].unique())))
        
        # Map string team names to integer indices (0 to N)
        team_to_idx = {team: idx for idx, team in enumerate(teams_list)}
        df_lg['home_idx'] = df_lg['home_team'].map(team_to_idx)
        df_lg['away_idx'] = df_lg['away_team'].map(team_to_idx)
        
        # Rename columns to match what `continuous_dixon_coles_nll` expects
        df_lg = df_lg.rename(columns={
            'home_xg': 'obs_xg_h',
            'away_xg': 'obs_xg_a',
            'home_goals': 'act_g_h',
            'away_goals': 'act_g_a'
        })
        
        try:
            # Run the SLSQP optimizer!
            prior_dict, meta_params = train_dixon_coles_prior(df_lg, teams_list)
            
            v4_priors[lg] = {
                "teams": prior_dict,
                "meta": meta_params
            }
            
            print(f"\n✅ {lg} Optimization Complete!")
            print(f"   Home Advantage (gamma): {meta_params['gamma_home_advantage']:.3f}")
            print(f"   Draw Correction (rho): {meta_params['rho_draw_correction']:.3f}")
            
            # Let's peek at the top 3 and bottom 3 attacking teams
            sorted_attacks = sorted(prior_dict.items(), key=lambda x: x[1]['alpha'], reverse=True)
            print(f"\n   Top 3 Attacks: {sorted_attacks[0][0]} ({sorted_attacks[0][1]['alpha']:.2f}), {sorted_attacks[1][0]} ({sorted_attacks[1][1]['alpha']:.2f}), {sorted_attacks[2][0]} ({sorted_attacks[2][1]['alpha']:.2f})")
            print(f"   Bot 3 Attacks: {sorted_attacks[-1][0]} ({sorted_attacks[-1][1]['alpha']:.2f}), {sorted_attacks[-2][0]} ({sorted_attacks[-2][1]['alpha']:.2f}), {sorted_attacks[-3][0]} ({sorted_attacks[-3][1]['alpha']:.2f})")
            
        except Exception as e:
            print(f"\n❌ Optimization failed for {lg}: {e}")



📈 Optimizing Parameters for: ENG-Premier League
Initiating Maximum Likelihood Estimation via SLSQP...
Optimization terminated successfully    (Exit mode 0)
            Current function value: 1146.3832369351655
            Iterations: 26
            Function evaluations: 1473
            Gradient evaluations: 26

✅ ENG-Premier League Optimization Complete!
   Home Advantage (gamma): 1.000
   Draw Correction (rho): -0.021

   Top 3 Attacks: Norwich City (1.00), Brighton (1.00), Crystal Palace (1.00)
   Bot 3 Attacks: Ipswich Town (1.00), Watford (1.00), Tottenham (1.00)

📈 Optimizing Parameters for: ESP-La Liga
Initiating Maximum Likelihood Estimation via SLSQP...
Optimization terminated successfully    (Exit mode 0)
            Current function value: 1149.418711786141
            Iterations: 34
            Function evaluations: 1855
            Gradient evaluations: 34

✅ ESP-La Liga Optimization Complete!
   Home Advantage (gamma): 0.998
   Draw Correction (rho): 0.034

   Top 3 Att

## 3. Save the V4 Priors
We save the finalized mathematically-pure parameters to a JSON file so they can be injected into the UI dashboard or a real-time betting script.

In [4]:
if v4_priors:
    out_path = Path("../v4_priors.json")
    with open(out_path, "w") as f:
        json.dump(v4_priors, f, indent=2)
    print(f"\n💾 All V4 Prior parameters successfully exported to {out_path}")



💾 All V4 Prior parameters successfully exported to ..\v4_priors.json
